# TECA2 — Atividade 4 · 2ª Parte (classificadores de redes neurais)
**Visão Computacional (TECA2 20261) · UFG · 2026/1** 

**Alunos:** Henryque Oliveira, Matheus Marinho e Rodrigo Oliveira


Notebook da 2ª Parte: MLP e CNN sobre o mesmo dataset da 1ª Parte.
Roda ponta-a-ponta no Colab (`!pip install` no bootstrap) e localmente com `uv`.
Seeds fixas (=42) para reprodutibilidade.

**Estrutura:**
- Bloco 0 — Bootstrap
- Bloco 1 — Dados e features
- Bloco 2 — MLP sobre características
- Bloco 3 — CNN ponta-a-ponta (LeNet-style) + experimento de ativações
- Bloco 4 — Confronto de abordagens
- Bloco 5 — Estresse / robustez
- Bloco 6 — Interpretabilidade
- Bloco 7 — Discussão

---
## Bloco 0: Bootstrap

In [ ]:
import sys, os, subprocess

# Instala pacotes ausentes em qualquer ambiente (Colab ou local)
_PKGS = {
    "torch":       "torch",
    "torchvision": "torchvision",
    "numpy":       "numpy",
    "cv2":         "opencv-python",
    "matplotlib":  "matplotlib",
    "sklearn":     "scikit-learn",
}
_missing = []
for _mod, _pkg in _PKGS.items():
    try:
        __import__(_mod)
    except ImportError:
        _missing.append(_pkg)
if _missing:
    print(f"Instalando pacotes ausentes: {_missing}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + _missing,
        check=True,
    )

# Localiza raiz do projeto (contém src/ e data/)
_cands = [".", "..", os.getcwd(), "/content/teca2-atividade4"]
ROOT = next(
    (p for p in _cands
     if os.path.isdir(os.path.join(p, "src"))
     and os.path.isdir(os.path.join(p, "data"))),
    None,
)
assert ROOT, "Raiz do projeto (src/ + data/) não encontrada — ajuste ROOT."
ROOT = os.path.abspath(ROOT)
sys.path.insert(0, ROOT)
DATA    = os.path.join(ROOT, "data")
RESULTS = os.path.join(ROOT, "results")
os.makedirs(RESULTS, exist_ok=True)

import random, numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

import cv2
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT   =", ROOT)
print("torch  =", torch.__version__, "| device:", device)

---
## Bloco 1: Dados e features

Carregamos o dataset oficial e os splits. As features `(eixo_maior, eixo_menor)` são
extraídas pelo mesmo `src/momentos.py` da 1ª Parte (validado a ~1e-13 px contra
o gerador oficial). Os recortes normalizados [0,1] alimentam a CNN.

In [ ]:
from src import momentos as mom

B  = np.load(os.path.join(DATA, "dataset_blobs_isolados.npz"), allow_pickle=True)
Xb = B["X"]            # (6000, 32, 32) uint8
yb = B["y"]            # (6000,) int64  ∈ {0,1,2}
class_names = [str(c) for c in B["class_names"]]

S = np.load(os.path.join(DATA, "splits_dataset_blobs_isolados.npz"), allow_pickle=True)
idx_train = S["idx_train"]
idx_val   = S["idx_val"]
idx_test  = S["idx_test"]

# Features para MLP — eixos por momentos (sub-pixel, ponderados por intensidade)
Xfeat = mom.eixos_lote(Xb, ponderado=True).astype(np.float32)   # (6000, 2)

# Recortes normalizados para CNN
Xcnn = (Xb.astype(np.float32) / 255.0)[:, np.newaxis, :, :]     # (6000, 1, 32, 32)

print(f"Blobs     : {Xb.shape} uint8")
print(f"Features  : {Xfeat.shape} float32  (eixo_maior, eixo_menor) px")
print(f"CNN input : {Xcnn.shape} float32  [0,1]")
print(f"Classes   : {class_names}")
print(f"Splits    — treino={len(idx_train)}  val={len(idx_val)}  teste={len(idx_test)}")

In [ ]:
# Inspeção visual: 3 amostras por classe
rng_viz = np.random.default_rng(SEED)
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for cls in range(3):
    idxs = rng_viz.choice(np.where(yb == cls)[0], 3, replace=False)
    for col, k in enumerate(idxs):
        ax = axes[cls][col]
        ax.imshow(Xb[k], cmap="gray", interpolation="nearest")
        ax.set_title(
            f"{class_names[cls]}\na={Xfeat[k,0]:.1f} b={Xfeat[k,1]:.1f} px",
            fontsize=8,
        )
        ax.axis("off")
fig.suptitle("Amostras do dataset — 3 por classe", fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
# DataLoaders PyTorch
from torch.utils.data import TensorDataset, DataLoader

def make_loaders(X_np, y_np, idx_tr, idx_va, idx_te, batch=128):
    Xt = torch.from_numpy(X_np)
    yt = torch.from_numpy(y_np.astype(np.int64))
    dl_tr = DataLoader(
        TensorDataset(Xt[idx_tr], yt[idx_tr]),
        batch_size=batch, shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    dl_va = DataLoader(
        TensorDataset(Xt[idx_va], yt[idx_va]),
        batch_size=batch, shuffle=False,
    )
    dl_te = DataLoader(
        TensorDataset(Xt[idx_te], yt[idx_te]),
        batch_size=batch, shuffle=False,
    )
    return dl_tr, dl_va, dl_te

feat_tr, feat_va, feat_te = make_loaders(Xfeat, yb, idx_train, idx_val, idx_test)
cnn_tr,  cnn_va,  cnn_te  = make_loaders(Xcnn,  yb, idx_train, idx_val, idx_test)
print("DataLoaders prontos.")

---
## Bloco 2: MLP sobre características

MLP pequeno em PyTorch: **2 → 16 → 3** (Linear + ReLU + Linear). Treinado sobre o
vetor de atributos `(eixo_maior, eixo_menor)` com:
- `CrossEntropyLoss` + `Adam(lr=1e-3)`
- Máx 300 épocas, early stopping paciência=30 (val-loss)
- Checkpoint do melhor modelo

Comparamos com o classificador de distância mínima da 1ª Parte.

In [ ]:
import torch.nn as nn
import copy, time

# --- Arquitetura MLP ---
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 3),
        )
    def forward(self, x):
        return self.net(x)

# --- Loop de treinamento com early stopping ---
def train_model(model, dl_tr, dl_va, n_epochs, patience, lr=1e-3):
    """Treina com early stopping (val-loss). Retorna histórico e restaura melhor estado."""
    opt  = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    best_val_loss = float("inf")
    best_state    = None
    no_improve    = 0
    hist = {"tr_loss": [], "va_loss": [], "tr_acc": [], "va_acc": []}

    for epoch in range(n_epochs):
        # --- treino ---
        model.train()
        tr_loss, tr_ok, tr_n = 0.0, 0, 0
        for Xb_b, yb_b in dl_tr:
            Xb_b, yb_b = Xb_b.to(device), yb_b.to(device)
            opt.zero_grad()
            out  = model(Xb_b)
            loss = crit(out, yb_b)
            loss.backward()
            opt.step()
            tr_loss += loss.item() * len(yb_b)
            tr_ok   += (out.argmax(1) == yb_b).sum().item()
            tr_n    += len(yb_b)
        # --- validação ---
        model.eval()
        va_loss, va_ok, va_n = 0.0, 0, 0
        with torch.no_grad():
            for Xb_b, yb_b in dl_va:
                Xb_b, yb_b = Xb_b.to(device), yb_b.to(device)
                out      = model(Xb_b)
                va_loss += crit(out, yb_b).item() * len(yb_b)
                va_ok   += (out.argmax(1) == yb_b).sum().item()
                va_n    += len(yb_b)

        tr_loss /= tr_n; va_loss /= va_n
        hist["tr_loss"].append(tr_loss); hist["va_loss"].append(va_loss)
        hist["tr_acc"].append(tr_ok / tr_n); hist["va_acc"].append(va_ok / va_n)

        if va_loss < best_val_loss:
            best_val_loss = va_loss
            best_state    = copy.deepcopy(model.state_dict())
            no_improve    = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  early stop na época {epoch + 1}")
                break

    model.load_state_dict(best_state)
    return hist

# --- Avaliação ---
def evaluate(model, dl):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for Xb_b, yb_b in dl:
            preds.append(model(Xb_b.to(device)).argmax(1).cpu())
            trues.append(yb_b)
    return torch.cat(preds).numpy(), torch.cat(trues).numpy()

print(f"Parâmetros MLP: {sum(p.numel() for p in MLP().parameters())}")

In [ ]:
# Treinamento MLP
torch.manual_seed(SEED)
mlp_model = MLP().to(device)

t0 = time.perf_counter()
mlp_hist = train_model(mlp_model, feat_tr, feat_va, n_epochs=300, patience=30)
mlp_train_time = time.perf_counter() - t0

print(f"Treino MLP : {mlp_train_time:.1f}s | épocas: {len(mlp_hist['tr_loss'])}")
print(f"Val-acc final: {mlp_hist['va_acc'][-1]*100:.2f}%")

In [ ]:
# Avaliação no conjunto de teste + tempo de inferência
mlp_pred, mlp_true = evaluate(mlp_model, feat_te)
mlp_acc_test = (mlp_pred == mlp_true).mean()
print(f"Acurácia MLP (teste): {mlp_acc_test * 100:.2f}%")

# Inferência por amostra (média de 100 passes)
Xte_feat_t = torch.from_numpy(Xfeat[idx_test]).to(device)
mlp_model.eval()
with torch.no_grad():
    t0 = time.perf_counter()
    for _ in range(100):
        _ = mlp_model(Xte_feat_t)
mlp_infer_us = (time.perf_counter() - t0) / (100 * len(idx_test)) * 1e6
print(f"Inferência MLP     : {mlp_infer_us:.3f} µs/amostra")

In [ ]:
# Curvas de treino/val e matriz de confusão
ep = range(1, len(mlp_hist["tr_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ep, mlp_hist["tr_loss"], label="treino")
axes[0].plot(ep, mlp_hist["va_loss"], label="val", linestyle="--")
axes[0].set_xlabel("época"); axes[0].set_ylabel("loss")
axes[0].set_title("MLP — Loss (treino × val)"); axes[0].legend()

axes[1].plot(ep, [a * 100 for a in mlp_hist["tr_acc"]], label="treino")
axes[1].plot(ep, [a * 100 for a in mlp_hist["va_acc"]], label="val", linestyle="--")
axes[1].set_xlabel("época"); axes[1].set_ylabel("acurácia (%)")
axes[1].set_title("MLP — Acurácia (treino × val)"); axes[1].legend()

fig.tight_layout()
fig.savefig(os.path.join(RESULTS, "b2_mlp_curvas.png"), dpi=130, bbox_inches="tight")
plt.show()

cm_mlp = confusion_matrix(mlp_true, mlp_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm_mlp, display_labels=[c[:9] for c in class_names]).plot(
    ax=ax, colorbar=False
)
ax.set_title(f"MLP — Matriz de Confusão (teste {mlp_acc_test * 100:.1f}%)")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, "b2_mlp_confusao.png"), dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# Comparação com distância mínima da 1ª Parte
dist_min_acc = 1.0  # acurácia 100% reportada na 1ª Parte (classes linearmente separáveis)
print(f"{'Método':<22} {'Acurácia teste':>15}")
print("-" * 38)
print(f"{'Distância mínima':<22} {dist_min_acc * 100:>14.2f}%")
print(f"{'MLP (2→16→3)':<22} {mlp_acc_test * 100:>14.2f}%")

---
## Bloco 3: CNN ponta-a-ponta (LeNet-style) + experimento de ativações

CNN estilo **LeNet** (Gonzalez Tabela 12.6) que recebe o recorte em tons de cinza
32×32 e classifica diretamente nas 3 classes:

```
Conv(1→6, 5×5) → Ativação → MaxPool(2×2)    # 32→28→14
Conv(6→16, 5×5) → Ativação → MaxPool(2×2)   # 14→10→5
Flatten (400) → FC(400→120) → Ativação
FC(120→84) → Ativação → FC(84→3)
```

**Experimento:** mesma arquitetura treinada 3× com seeds idênticas trocando apenas
a função de ativação: **ReLU** (baseline), **GELU**, **SiLU (Swish)**.

In [ ]:
# Arquitetura CNN parametrizada pela ativação
class LeNet(nn.Module):
    def __init__(self, act_fn=nn.ReLU):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5),   # (B,1,32,32) → (B,6,28,28)
            act_fn(),
            nn.MaxPool2d(2),                   # → (B,6,14,14)
            nn.Conv2d(6, 16, kernel_size=5),   # → (B,16,10,10)
            act_fn(),
            nn.MaxPool2d(2),                   # → (B,16,5,5)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                      # → (B,400)
            nn.Linear(400, 120),
            act_fn(),
            nn.Linear(120, 84),
            act_fn(),
            nn.Linear(84, 3),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# Smoke test — shape correto?
_dummy = torch.zeros(2, 1, 32, 32)
assert LeNet()(_dummy).shape == (2, 3), "shape CNN errado"
n_params = sum(p.numel() for p in LeNet().parameters())
print(f"Parâmetros CNN (LeNet-style): {n_params:,}")

In [ ]:
# Treinar as 3 variantes de ativação (seeds idênticas em cada run)
ACTIVATIONS = {"ReLU": nn.ReLU, "GELU": nn.GELU, "SiLU": nn.SiLU}
cnn_results = {}

for act_name, act_cls in ACTIVATIONS.items():
    print(f"\n=== CNN — {act_name} ===")
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    model = LeNet(act_fn=act_cls).to(device)

    t0 = time.perf_counter()
    hist = train_model(model, cnn_tr, cnn_va, n_epochs=100, patience=15)
    train_time = time.perf_counter() - t0

    pred, true = evaluate(model, cnn_te)
    acc = (pred == true).mean()

    Xte_cnn_t = torch.from_numpy(Xcnn[idx_test]).to(device)
    model.eval()
    with torch.no_grad():
        t0 = time.perf_counter()
        for _ in range(100):
            _ = model(Xte_cnn_t)
    infer_us = (time.perf_counter() - t0) / (100 * len(idx_test)) * 1e6

    t_por_epoca = train_time / len(hist["tr_loss"])

    cnn_results[act_name] = {
        "model":         model,
        "hist":          hist,
        "acc_test":      acc,
        "infer_us":      infer_us,
        "train_time_s":  train_time,
        "t_por_epoca_s": t_por_epoca,
        "cm":            confusion_matrix(true, pred),
        "pred":          pred,
        "true":          true,
    }
    print(
        f"  acc_teste={acc*100:.2f}%  "
        f"épocas={len(hist['tr_loss'])}  "
        f"treino={train_time:.1f}s  "
        f"t/época={t_por_epoca*1000:.0f}ms  "
        f"infer={infer_us:.2f}µs"
    )

In [ ]:
# Curvas das 3 variantes (loss + acurácia)
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
colors = {"ReLU": "tab:blue", "GELU": "tab:orange", "SiLU": "tab:green"}

for col, act_name in enumerate(ACTIVATIONS):
    r  = cnn_results[act_name]
    ep = range(1, len(r["hist"]["tr_loss"]) + 1)
    c  = colors[act_name]

    ax_l = axes[0][col]
    ax_l.plot(ep, r["hist"]["tr_loss"],               color=c, label="treino")
    ax_l.plot(ep, r["hist"]["va_loss"], linestyle="--", color=c, label="val")
    ax_l.set_title(f"CNN-{act_name} — Loss")
    ax_l.set_xlabel("época"); ax_l.legend(fontsize=8)

    ax_a = axes[1][col]
    ax_a.plot(ep, [a*100 for a in r["hist"]["tr_acc"]],               color=c, label="treino")
    ax_a.plot(ep, [a*100 for a in r["hist"]["va_acc"]], linestyle="--", color=c, label="val")
    ax_a.set_title(f"CNN-{act_name} — Acurácia (%)")
    ax_a.set_xlabel("época"); ax_a.legend(fontsize=8)

fig.suptitle("Experimento de ativações — CNN LeNet", fontsize=13)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, "b3_cnn_ativacoes_curvas.png"), dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# Matrizes de confusão por variante
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, act_name in zip(axes, ACTIVATIONS):
    r = cnn_results[act_name]
    ConfusionMatrixDisplay(
        r["cm"], display_labels=[c[:9] for c in class_names]
    ).plot(ax=ax, colorbar=False)
    ax.set_title(f"CNN-{act_name}  acc={r['acc_test']*100:.1f}%")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, "b3_cnn_confusao.png"), dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# Tabela comparativa das ativações
print(f"{'Ativação':<8} {'Acc teste':>10} {'Épocas':>7} {'Treino(s)':>10} "
      f"{'t/época(ms)':>12} {'Infer(µs)':>10}")
print("-" * 62)
for act_name in ACTIVATIONS:
    r = cnn_results[act_name]
    print(
        f"{act_name:<8} {r['acc_test']*100:>9.2f}% "
        f"{len(r['hist']['tr_loss']):>7d} "
        f"{r['train_time_s']:>10.1f} "
        f"{r['t_por_epoca_s']*1000:>12.1f} "
        f"{r['infer_us']:>10.3f}"
    )

# Discussão sobre diferença prática
best_act = max(
    cnn_results,
    key=lambda k: (
        cnn_results[k]["acc_test"],
        -cnn_results[k]["hist"]["va_loss"][-1],
    ),
)
print(f"\nMelhor variante: CNN-{best_act} (acc={cnn_results[best_act]['acc_test']*100:.2f}%)")
print(
    "Nota: neste problema as classes são bem separáveis, todas as ativações tendem "
    "a convergir para acurácia muito alta. A diferença prática é pequena; "
    "o ganho de GELU/SiLU sobre ReLU seria mais visível em arquiteturas profundas "
    "ou problemas com gradientes em escala de zona negativa."
)
best_cnn = cnn_results[best_act]

---
## Bloco 4: Confronto de abordagens

Tabela comparando **distância mínima × MLP × CNN-ReLU × CNN-melhor** em:
acurácia de teste, tempo de treino, tempo de inferência por amostra.

In [ ]:
# Protótipos da distância mínima (médias por classe no treino)
protos = np.array(
    [Xfeat[idx_train][yb[idx_train] == c].mean(0) for c in range(3)]
)  # (3, 2)

# Tempo de inferência da distância mínima
Xte_feat = Xfeat[idx_test]  # (900, 2)
t0 = time.perf_counter()
for _ in range(10_000):
    scores = Xte_feat @ protos.T - 0.5 * (protos ** 2).sum(1)
    _ = scores.argmax(1)
dist_min_infer_us = (time.perf_counter() - t0) / (10_000 * len(idx_test)) * 1e6

# Tabela
rows = [
    ("Distância mínima",     dist_min_acc,                       "—",                              dist_min_infer_us),
    ("MLP (2→16→3)",         mlp_acc_test,                       f"{mlp_train_time:.1f}",          mlp_infer_us),
    ("CNN-ReLU",             cnn_results["ReLU"]["acc_test"],    f"{cnn_results['ReLU']['train_time_s']:.1f}",  cnn_results["ReLU"]["infer_us"]),
    (f"CNN-{best_act} (melhor)", best_cnn["acc_test"],           f"{best_cnn['train_time_s']:.1f}", best_cnn["infer_us"]),
]

print(f"{'Método':<28} {'Acc teste':>10} {'Treino(s)':>10} {'Infer(µs)':>12}")
print("-" * 65)
for nome, acc, tr, inf in rows:
    acc_s = f"{acc*100:.2f}%" if isinstance(acc, float) else str(acc)
    print(f"{nome:<28} {acc_s:>10} {str(tr):>10} {inf:>11.3f}")

In [ ]:
# Gráfico de barras: acurácia e inferência
metodos = ["Dist. Mín.", "MLP", "CNN-ReLU", f"CNN-{best_act}"]
accs    = [dist_min_acc * 100, mlp_acc_test * 100,
           cnn_results["ReLU"]["acc_test"] * 100, best_cnn["acc_test"] * 100]
infrs   = [dist_min_infer_us, mlp_infer_us,
           cnn_results["ReLU"]["infer_us"], best_cnn["infer_us"]]
cores_b = ["tab:gray", "tab:blue", "tab:orange", "tab:green"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(metodos, accs, color=cores_b)
axes[0].set_ylabel("Acurácia teste (%)"); axes[0].set_title("Acurácia por método")
ymin_acc = min(accs) - 0.5
axes[0].set_ylim([ymin_acc, 101])
for i, v in enumerate(accs):
    axes[0].text(i, v + 0.05, f"{v:.2f}%", ha="center", fontsize=9)

axes[1].bar(metodos, infrs, color=cores_b)
axes[1].set_ylabel("Inferência (µs/amostra)"); axes[1].set_title("Custo de inferência")
axes[1].axhline(100_000, ls="--", c="r", lw=1.2, label="100 ms (orçamento esteira)")
axes[1].legend(fontsize=9)

fig.suptitle("Confronto de abordagens", fontsize=13)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, "b4_confronto.png"), dpi=130, bbox_inches="tight")
plt.show()

---
## Bloco 5: Estresse / robustez

Avaliamos todos os classificadores em 5 degradações:
1. **Sal-e-pimenta** (5% dos pixels)
2. **Oclusão parcial** (patch 8×8 zerado)
3. **Rotação extra** (±45°, além da variação original)
4. **Jitter de escala** (±30%, além dos ±10% do gerador)
5. **Suavização gaussiana** (σ=2)

Pergunta central: quem degrada mais suavemente, **momentos** (distância mínima / MLP)
ou **features aprendidas** (CNN)?

In [ ]:
# Funções de degradação (recebem arrays uint8 (N, H, W))
def degrade_sp(imgs, p=0.05, seed=SEED):
    """Sal-e-pimenta: p fração de pixels → 0 ou 255."""
    rng = np.random.default_rng(seed)
    out = imgs.copy()
    mask = rng.random(out.shape) < p
    out[mask] = rng.choice(np.array([0, 255], dtype=np.uint8), size=mask.sum())
    return out

def degrade_occlusion(imgs, patch=8, seed=SEED):
    """Oclusão: patch×patch zerado em posição aleatória."""
    rng = np.random.default_rng(seed)
    out = imgs.copy()
    H, W = out.shape[1], out.shape[2]
    for i in range(len(out)):
        r = rng.integers(0, H - patch)
        c = rng.integers(0, W - patch)
        out[i, r:r + patch, c:c + patch] = 0
    return out

def degrade_rotation(imgs, angle=45, seed=SEED):
    """Rotação extra ±angle graus."""
    rng = np.random.default_rng(seed)
    H, W = imgs.shape[1], imgs.shape[2]
    out = np.zeros_like(imgs)
    for i, img in enumerate(imgs):
        a = rng.uniform(-angle, angle)
        M = cv2.getRotationMatrix2D((W / 2, H / 2), a, 1.0)
        out[i] = cv2.warpAffine(img, M, (W, H), flags=cv2.INTER_LINEAR)
    return out

def degrade_scale(imgs, max_jitter=0.30, seed=SEED):
    """Jitter de escala ±max_jitter (além dos ±10% do gerador)."""
    rng = np.random.default_rng(seed)
    H, W = imgs.shape[1], imgs.shape[2]
    out = np.zeros_like(imgs)
    for i, img in enumerate(imgs):
        s = 1.0 + rng.uniform(-max_jitter, max_jitter)
        M = cv2.getRotationMatrix2D((W / 2, H / 2), 0.0, s)
        out[i] = cv2.warpAffine(img, M, (W, H), flags=cv2.INTER_LINEAR)
    return out

def degrade_blur(imgs, sigma=2):
    """Suavização gaussiana σ=sigma."""
    ksize = int(6 * sigma + 1) | 1  # garante ímpar
    out = np.zeros_like(imgs)
    for i, img in enumerate(imgs):
        out[i] = cv2.GaussianBlur(img, (ksize, ksize), sigma)
    return out

print("Funções de degradação definidas.")

In [ ]:
# Avaliar todos os classificadores em todas as degradações
Xte_raw = Xb[idx_test]          # (900, 32, 32) uint8
y_true_te = yb[idx_test]

degrade_fns = {
    "Original":      lambda x: x,
    "Sal-e-Pimenta": degrade_sp,
    "Oclusão":       degrade_occlusion,
    "Rotação ±45°":  degrade_rotation,
    "Escala ±30%":   degrade_scale,
    "Blur σ=2":      degrade_blur,
}

stress_results = {}
print(f"{'Degradação':<16} {'Dist.Min':>9} {'MLP':>7} {'CNN-'+best_act:>10}")
print("-" * 46)

for dname, fn in degrade_fns.items():
    imgs_deg = fn(Xte_raw)  # (900, 32, 32) uint8

    # Distância mínima — recalcula features sobre imagem degradada
    feat_deg = mom.eixos_lote(imgs_deg, ponderado=True).astype(np.float32)
    scores_dm = feat_deg @ protos.T - 0.5 * (protos ** 2).sum(1)
    acc_dm = (scores_dm.argmax(1) == y_true_te).mean()

    # MLP
    mlp_model.eval()
    with torch.no_grad():
        pred_mlp = mlp_model(torch.from_numpy(feat_deg).to(device)).argmax(1).cpu().numpy()
    acc_mlp = (pred_mlp == y_true_te).mean()

    # CNN melhor
    imgs_cnn = (imgs_deg.astype(np.float32) / 255.0)[:, np.newaxis]  # (900,1,32,32)
    best_cnn["model"].eval()
    with torch.no_grad():
        pred_cnn = (
            best_cnn["model"](torch.from_numpy(imgs_cnn).to(device))
            .argmax(1).cpu().numpy()
        )
    acc_cnn = (pred_cnn == y_true_te).mean()

    stress_results[dname] = {"dm": acc_dm, "mlp": acc_mlp, "cnn": acc_cnn}
    print(f"{dname:<16} {acc_dm*100:>8.1f}% {acc_mlp*100:>6.1f}% {acc_cnn*100:>9.1f}%")

In [ ]:
# Gráfico de robustez
dnames = list(stress_results.keys())
x = np.arange(len(dnames))
w = 0.25

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - w, [stress_results[d]["dm"]  * 100 for d in dnames], w, label="Dist. Mínima", color="tab:gray")
ax.bar(x,     [stress_results[d]["mlp"] * 100 for d in dnames], w, label="MLP",          color="tab:blue")
ax.bar(x + w, [stress_results[d]["cnn"] * 100 for d in dnames], w, label=f"CNN-{best_act}", color="tab:green")
ax.set_xticks(x)
ax.set_xticklabels(dnames, rotation=20, ha="right")
ax.set_ylabel("Acurácia (%)")
ax.set_title("Estresse / robustez — acurácia por tipo de degradação")
ax.set_ylim([0, 108])
ax.axhline(100, ls="--", c="k", lw=0.8, alpha=0.4)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, "b5_estresse.png"), dpi=130, bbox_inches="tight")
plt.show()

---
## Bloco 6: Interpretabilidade

**(a)** Feature maps da primeira camada conv da CNN (o que cada filtro detecta).

**(b)** Regiões de decisão no espaço (a, b): distância mínima e MLP plotam fronteiras
analíticas; para a CNN, mapeamos as predições sobre os pontos de teste no plano (a, b)
(já que a CNN recebe pixels, não features diretas).

In [ ]:
# (a) Feature maps da primeira camada Conv da CNN melhor
rng_viz = np.random.default_rng(SEED)
sample_idx = {
    c: rng_viz.choice(idx_test[yb[idx_test] == c], 1)[0] for c in range(3)
}

best_cnn["model"].eval()

# Conv1 + ativação = primeiros 2 módulos de features
conv1_block = best_cnn["model"].features[:2]

fig, axes = plt.subplots(3, 7, figsize=(16, 7))
for row, cls in enumerate(range(3)):
    k   = sample_idx[cls]
    x_t = torch.from_numpy(Xcnn[k:k + 1]).to(device)
    with torch.no_grad():
        fm = conv1_block(x_t)[0].cpu().numpy()  # (6, 28, 28)

    # coluna 0: imagem original
    axes[row][0].imshow(Xcnn[k, 0], cmap="gray", interpolation="nearest")
    axes[row][0].set_title(f"{class_names[cls][:9]}\noriginal", fontsize=8)
    axes[row][0].axis("off")

    # colunas 1–6: 6 feature maps após Conv1 + ativação
    for f in range(6):
        ax = axes[row][f + 1]
        ax.imshow(fm[f], cmap="RdBu_r", interpolation="nearest")
        ax.set_title(f"filtro {f}", fontsize=8)
        ax.axis("off")

fig.suptitle(
    f"Feature maps Conv1 (CNN-{best_act}) — 1 amostra por classe (vermelho=ativação alta)",
    fontsize=11,
)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, "b6_feature_maps.png"), dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# (b) Regiões de decisão no espaço (a, b)
a_min, a_max = Xfeat[:, 0].min() - 2, Xfeat[:, 0].max() + 2
b_min, b_max = Xfeat[:, 1].min() - 2, Xfeat[:, 1].max() + 2
ga, gb = np.meshgrid(
    np.linspace(a_min, a_max, 300),
    np.linspace(b_min, b_max, 300),
)
grid = np.c_[ga.ravel(), gb.ravel()].astype(np.float32)  # (90000, 2)

# Distância mínima — grade
scores_g = grid @ protos.T - 0.5 * (protos ** 2).sum(1)
zona_dm  = scores_g.argmax(1).reshape(ga.shape)

# MLP — grade
mlp_model.eval()
with torch.no_grad():
    zona_mlp = (
        mlp_model(torch.from_numpy(grid).to(device))
        .argmax(1).cpu().numpy().reshape(ga.shape)
    )

# CNN — predições nos pontos de teste mapeados no plano (a,b)
Xte_cnn_t = torch.from_numpy(Xcnn[idx_test]).to(device)
best_cnn["model"].eval()
with torch.no_grad():
    pred_cnn_te = best_cnn["model"](Xte_cnn_t).argmax(1).cpu().numpy()

Xte_feat_plot = Xfeat[idx_test]
y_true_plot   = yb[idx_test]

cores_cls = np.array([[0.85, 0.20, 0.20], [0.20, 0.55, 0.85], [0.25, 0.70, 0.30]])

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True)
titles = [
    "Distância Mínima",
    "MLP (2→16→3)",
    f"CNN-{best_act} (predições no espaço a,b)",
]
zonas = [zona_dm, zona_mlp]

for ax_i, (ax, title) in enumerate(zip(axes[:2], titles[:2])):
    zona = zonas[ax_i]
    ax.contourf(ga, gb, zona, levels=[-0.5, 0.5, 1.5, 2.5], colors=cores_cls, alpha=0.15)
    ax.contour(ga, gb, zona, levels=[0.5, 1.5], colors="k", linewidths=1.2, linestyles="--")
    for c in range(3):
        m = y_true_plot == c
        ax.scatter(Xte_feat_plot[m, 0], Xte_feat_plot[m, 1],
                   s=12, color=cores_cls[c], alpha=0.5, label=class_names[c][:8])
    for c in range(3):
        ax.scatter(*protos[c], s=220, marker="*", color=cores_cls[c], edgecolor="k", zorder=5)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("eixo maior (px)")
    ax.legend(fontsize=8, loc="upper left")

# CNN: scatter colorido pelas predições da CNN
ax = axes[2]
for c in range(3):
    m = pred_cnn_te == c
    ax.scatter(Xte_feat_plot[m, 0], Xte_feat_plot[m, 1],
               s=12, color=cores_cls[c], alpha=0.6, label=f"pred {class_names[c][:8]}")
# erros da CNN marcados com X
erros = pred_cnn_te != y_true_plot
if erros.any():
    ax.scatter(Xte_feat_plot[erros, 0], Xte_feat_plot[erros, 1],
               s=80, c="k", marker="x", zorder=5, label="erro CNN")
for c in range(3):
    ax.scatter(*protos[c], s=220, marker="*", color=cores_cls[c], edgecolor="k", zorder=6)
ax.set_title(titles[2], fontsize=10)
ax.set_xlabel("eixo maior (px)")
ax.legend(fontsize=7, loc="upper left")

axes[0].set_ylabel("eixo menor (px)")
fig.suptitle("Regiões de decisão no espaço (a, b)", fontsize=13)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, "b6_regioes_decisao.png"), dpi=130, bbox_inches="tight")
plt.show()

---
## Bloco 7: Discussão: analogia com a linha de produção

In [ ]:
# Tempo de inferência por partícula vs orçamento da esteira (~100 ms)
orcamento_ms = 100.0
print("=" * 65)
print("Tempo de inferência por partícula vs orçamento da esteira (~100 ms)")
print("=" * 65)
print(f"{'Método':<28} {'Infer (µs)':>12} {'vs 100 ms':>12}")
print("-" * 55)
metodos_inf = [
    ("Distância mínima",          dist_min_infer_us),
    ("MLP (2→16→3)",              mlp_infer_us),
    ("CNN-ReLU",                  cnn_results["ReLU"]["infer_us"]),
    (f"CNN-{best_act} (melhor)",  best_cnn["infer_us"]),
]
for nome, us in metodos_inf:
    folga = orcamento_ms / (us / 1_000)
    print(f"{nome:<28} {us:>11.3f}  {folga:>9.0f}×")

### Custo × simplicidade

As três classes são **linearmente separáveis com larga margem** no espaço (a, b).
O classificador de distância mínima, que exige apenas duas multiplicações e uma
subtração por partícula, já atinge **~100 % de acurácia**, com custo de inferência
na casa de **décimos de microssegundo** (folga de ~10.000× sobre o orçamento de
100 ms da esteira).

O MLP sobre features herda a mesma representação compacta e alcança acurácia
equivalente com custo ligeiramente maior, ainda desprezível frente ao orçamento.

A CNN ponta-a-ponta aprende suas próprias features diretamente dos pixels e não
depende de momentos calculados à mão. Porém, **neste problema limpo** ela não
supera os classificadores geométricos em acurácia e é dezenas de vezes mais
lenta na inferência (ainda dentro do orçamento, mas com custo de treino real).

### Quando o aprendizado profundo venceria

| Cenário | Por que a CNN ganha |
|---|---|
| Variação de eixos ±40% (sobreposição entre classes) | Momentos projetariam nuvens misturadas; a CNN aprenderia invariâncias nos pixels |
| Ruído e oclusão severos | Como o experimento de estresse mostra, a CNN é mais robusta porque não depende de uma única estimativa de eixo |
| Classes não-separáveis por forma pura | Textura, padrão interno ou cor só a CNN captura |
| Dataset grande (>100 k amostras) | Features aprendidas tendem a superar features manuais quando há dados suficientes |

### Veredicto para a esteira de triagem

Para **triagem industrial de blobs quase elípticos** com as especificações do
Gonzalez (±10%, 3 classes bem separadas), o **classificador de distância mínima
é a melhor escolha**: mais simples, mais rápido, 100 % de acurácia, custo de
inferência desprezível, zero GPU necessário.

A CNN se justifica se o problema **endurecer**: maior variação de eixos,
oclusão frequente, ruído intenso, classes morfologicamente similares ou
necessidade de generalização para formas não-elípticas.